# Election Noise: pull Google political ad spend

In [1]:
PROJECT = "election-noise"

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

In [2]:
raw_d = client.query("""
  SELECT country_subdivision_primary AS st,
         country_subdivision_secondary AS district,
         spend_usd
  FROM `bigquery-public-data.google_political_ads.geo_spend`
  WHERE country = 'US'
""").to_dataframe()
print(len(raw_d), 'rows')
raw_d.sample(10, random_state=1)

1134 rows


,st,district,spend_usd
1099,US-WA,WA-2 (2022 redistricting),1975100
774,US-NY,NY-20 (2024 redistricting),1983000
385,US-ID,ID-2 (2022 redistricting),2686300
94,US-CA,CA-30 (2012 redistricting),1842500
403,US-IL,IL-5 (2012 redistricting),1651900
935,US-TN,TN-8 (2022 redistricting),827800
244,US-DC,DC-AT LARGE (2012 redistricting),4246900
917,US-TN,TN-9 (2026 redistricting),1092500
331,US-GA,GA-8 (2022 redistricting),1507300
776,US-NY,NY-27 (2012 redistricting),644500


In [3]:
import re
import pandas as pd

ABBR_TO_FIPS = {
    "AL":"01","AK":"02","AZ":"04","AR":"05","CA":"06","CO":"08","CT":"09","DE":"10","DC":"11",
    "FL":"12","GA":"13","HI":"15","ID":"16","IL":"17","IN":"18","IA":"19","KS":"20","KY":"21",
    "LA":"22","ME":"23","MD":"24","MA":"25","MI":"26","MN":"27","MS":"28","MO":"29","MT":"30",
    "NE":"31","NV":"32","NH":"33","NJ":"34","NM":"35","NY":"36","NC":"37","ND":"38","OH":"39",
    "OK":"40","OR":"41","PA":"42","RI":"44","SC":"45","SD":"46","TN":"47","TX":"48","UT":"49",
    "VT":"50","VA":"51","WA":"53","WV":"54","WI":"55","WY":"56",
}

AT_LARGE = {"AK","DE","ND","SD","VT","WY"}

def state_fips(primary):
    """'US-CA' -> '06'"""
    if not isinstance(primary, str):
        return None
    return ABBR_TO_FIPS.get(primary.strip().upper().split("-")[-1])

def district_geoid(primary, secondary):
    """('US-CA', 'CA-12' / 'California 12th' / ...) -> '0612'"""
    fips = state_fips(primary)
    if fips is None or not isinstance(secondary, str) or not secondary.strip():
        return None
    abbr = primary.strip().upper().split("-")[-1]
    if abbr == "DC":
        return fips + "98"
    if abbr in AT_LARGE or re.search(r"at[\s-]?large|-AL\b", secondary, re.I):
        return fips + "00"
    nums = re.findall(r"\d+", secondary)
    if not nums:
        return None
    return fips + nums[-1].zfill(2)[-2:]


In [4]:
# quick sanity check on the key builder
for p, s in [("US-CA","CA-12"), ("US-PA","Pennsylvania's 3rd congressional district"),
             ("US-WY","WY-01"), ("US-MT","MT-AL"), ("US-DC","DC-00"), ("US-TX","")]:
    print(f"{p:6} {s!r:48} -> {district_geoid(p, s)}")


US-CA  'CA-12'                                          -> 0612
US-PA  "Pennsylvania's 3rd congressional district"      -> 4203
US-WY  'WY-01'                                          -> 5600
US-MT  'MT-AL'                                          -> 3000
US-DC  'DC-00'                                          -> 1198
US-TX  ''                                               -> None


In [5]:
raw_d['geoid'] = [district_geoid(p, s) for p, s in zip(raw_d.st, raw_d.district)]
unmatched = raw_d[raw_d.geoid.isna()]
print(len(unmatched), 'rows without a district key (state-level or unparsed):')
unmatched.head(15)

6 rows without a district key (state-level or unparsed):


,st,district,spend_usd,geoid
0,None,United States,61290300,None
1,AS,American Samoa,6200,None
2,GU,Guam,145300,None
3,MP,Northern Mariana Islands,9600,None
4,PR,Puerto Rico,337000,None
1133,VI,U.S. Virgin Islands,84300,None


In [6]:
districts = (raw_d.dropna(subset=['geoid'])
                .groupby('geoid', as_index=False)['spend_usd'].sum()
                .sort_values('spend_usd', ascending=False))
print(len(districts), 'districts (expect ~436 incl. DC)')
districts.head(10)

112 districts (expect ~436 incl. DC)


,geoid,spend_usd
10,0622,188859300
49,2622,185870600
87,4222,176796400
11,0626,134899400
110,5522,121753500
98,4826,112184100
23,1324,106616100
6,0422,99879100
81,3926,97996300
9,0612,82944700


`advertiser_geo_spend` is reported per state, which catches ads that weren't tied to one district.

In [7]:
raw_s = client.query("""
  SELECT country_subdivision_primary AS st, SUM(spend_usd) AS spend_usd
  FROM `bigquery-public-data.google_political_ads.advertiser_geo_spend`
  WHERE country = 'US'
  GROUP BY st
""").to_dataframe()
raw_s['state_fips'] = raw_s.st.map(state_fips)
raw_s['state'] = raw_s.st.str.split('-').str[-1]
states = (raw_s.dropna(subset=['state_fips'])[['state_fips','state','spend_usd']]
               .sort_values('spend_usd', ascending=False))
states.head(10)

,state_fips,state,spend_usd
18,06,CA,407442400
43,42,PA,225785800
25,26,MI,221596700
39,48,TX,217530700
19,13,GA,199192000
24,39,OH,185058800
46,12,FL,169411600
41,55,WI,146319500
49,37,NC,136510000
38,04,AZ,136171400


In [8]:
districts.to_csv('districts.csv', index=False)
states.to_csv('states.csv', index=False)

from google.colab import files
files.download('districts.csv')
files.download('states.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2026 midterms only, by state

Uses `creative_stats`, where each row is one ad with its dates, a spend range, and where it was targeted. Ads that started between Jan 1 2025 and Election Day count.

In [9]:
CYCLE_START = "2025-01-01"   # start of the 2025-26 cycle
CYCLE_END   = "2026-11-03"   # Election Day

ads = client.query(f"""
SELECT
  ad_id,
  advertiser_name,
  date_range_start,
  geo_targeting_included,
  # Google reports spend per ad as a range; use the midpoint
  # the top bucket has no max, so fall back to the min
  (spend_range_min_usd + COALESCE(spend_range_max_usd, spend_range_min_usd)) / 2 AS spend_est
FROM `bigquery-public-data.google_political_ads.creative_stats`
WHERE regions LIKE '%US%'
  AND date_range_start BETWEEN DATE '{CYCLE_START}' AND DATE '{CYCLE_END}'
""").to_dataframe()
print(len(ads), "ads,", f"${ads.spend_est.sum():,.0f} estimated total")
ads[["advertiser_name", "date_range_start", "geo_targeting_included", "spend_est"]].sample(10, random_state=1)


202875 ads, $1,345,035,600 estimated total


,advertiser_name,date_range_start,geo_targeting_included,spend_est
177057,Tea Party Patriots Action,2026-10-16,United States,50.0
17403,MAGA INC.,2026-10-26,"23301,Virginia,United States, 23302,Virginia,U...",50.0
87950,Lower Costs Now,2026-10-06,"Alger County,Michigan,United States, Crawford ...",7500.0
35080,HALEY STEVENS FOR SENATE,2026-07-30,"48021,Michigan,United States, 48033,Michigan,U...",47500.0
181462,WOWTOPIA LLC,2025-08-07,United States,50.0
110661,Village for Jamell Henderson,2026-07-30,"East Hartford,Connecticut,United States",850.0
115468,NO GOING BACK PAC INC.,2026-09-30,"Georgia,United States",27500.0
75016,Brian Pacheco for Assembly 2026,2026-10-13,"93210,California,United States, 93234,Californ...",2250.0
72606,"NO ON PROP 40 - DOCTORS, TEACHERS, SMALL BUSIN...",2026-10-30,"90703,California,United States, 91006,Californ...",50.0
145553,OHIOANS FOR LOWER COSTS,2026-09-02,"Ohio,United States",22500.0


In [10]:

NAME_TO_ABBR = {
 "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA","Colorado":"CO",
 "Connecticut":"CT","Delaware":"DE","District of Columbia":"DC","Florida":"FL","Georgia":"GA",
 "Hawaii":"HI","Idaho":"ID","Illinois":"IL","Indiana":"IN","Iowa":"IA","Kansas":"KS","Kentucky":"KY",
 "Louisiana":"LA","Maine":"ME","Maryland":"MD","Massachusetts":"MA","Michigan":"MI","Minnesota":"MN",
 "Mississippi":"MS","Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV","New Hampshire":"NH",
 "New Jersey":"NJ","New Mexico":"NM","New York":"NY","North Carolina":"NC","North Dakota":"ND",
 "Ohio":"OH","Oklahoma":"OK","Oregon":"OR","Pennsylvania":"PA","Rhode Island":"RI",
 "South Carolina":"SC","South Dakota":"SD","Tennessee":"TN","Texas":"TX","Utah":"UT","Vermont":"VT",
 "Virginia":"VA","Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY",
}

def targeted_states(s):
    if not isinstance(s, str):
        return []
    parts = [p.strip() for p in re.split(r"[;|,\n]", s)]
    found = []
    for i, p in enumerate(parts):
        abbr = NAME_TO_ABBR.get(p)
        if abbr is None:
            continue

        if abbr == "WA" and i + 1 < len(parts) and parts[i + 1] == "District of Columbia":
            continue
        if abbr not in found:
            found.append(abbr)
    return found

for t in ["California, United States", "Kansas City, Missouri, United States",
          "Washington, District of Columbia, United States", "Ohio; Texas", "United States"]:
    print(f"{t!r:52} -> {targeted_states(t)}")


'California, United States'                          -> ['CA']
'Kansas City, Missouri, United States'               -> ['MO']
'Washington, District of Columbia, United States'    -> ['DC']
'Ohio; Texas'                                        -> ['OH', 'TX']
'United States'                                      -> []


In [11]:

ads["states"] = ads.geo_targeting_included.map(targeted_states)
national = ads[ads.states.str.len() == 0]
print(f"left out as nationwide/unparsed: {len(national)} ads, ${national.spend_est.sum():,.0f}")

rows = [(st, r.spend_est / len(r.states))
        for r in ads[ads.states.str.len() > 0].itertuples() for st in r.states]
states_26 = (pd.DataFrame(rows, columns=["state", "spend_usd"])
               .groupby("state", as_index=False)["spend_usd"].sum())
states_26["state_fips"] = states_26.state.map(ABBR_TO_FIPS)
states_26 = states_26[["state_fips", "state", "spend_usd"]].sort_values("spend_usd", ascending=False)
states_26.head(15)


left out as nationwide/unparsed: 50309 ads, $191,335,000


,state_fips,state,spend_usd
4,06,CA,1.679039e+08
43,48,TX,1.155715e+08
35,39,OH,1.029385e+08
22,26,MI,8.061510e+07
12,19,IA,5.673537e+07
10,13,GA,5.120794e+07
9,12,FL,4.539054e+07
45,51,VA,4.098019e+07
48,55,WI,3.788209e+07
34,36,NY,3.583952e+07


In [13]:
states_26.to_csv("states.csv", index=False)
from google.colab import files
files.download("states.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>